In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()
if os.environ['GOOGLE_API_KEY']:
    print("API key is set.")
else:
    raise ValueError("API key is not set")

API key is set.


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal
from langchain.messages import SystemMessage, HumanMessage

In [ ]:
llm = ChatGoogleGenerativeAI(model = "gemini-3.5-flash-lite")

In [ ]:
class Graph_schema(BaseModel):
    query: str
    category: str
    response : str

class Category_schema(BaseModel):
    category: Literal["Technical","Billing","General"] = Field(description="classification for the users issue query for support")

In [ ]:
def classifier(state: Graph_schema)-> dict:
    query = state.query
    classifier = llm.with_structured_output(Category_schema).invoke(query)
    return {"category": classifier.category}

In [ ]:
def billing(state: Graph_schema) -> dict:
    return{"response": "pls contact the billing department"}
def technical(state: Graph_schema) -> dict:
    return{"response": "pls contact the technical department"}
def general(state: Graph_schema) -> dict:
    return{"response": "pls contact the customer support"}

In [ ]:
def conditional_call(state: Graph_schema) -> str:
    category = state.category
    if category == "Technical": return "technical"
    elif category == "Billing" : return "billing"
    elif category == "General" : return "general"

In [ ]:
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image,display

builder = StateGraph(Graph_schema)

builder.add_node("classifier", classifier)
builder.add_node("billing", billing)
builder.add_node("technical", technical)
builder.add_node("general", general)

builder.add_edge(START, "classifier")
builder.add_conditional_edges(
    "classifier",
    conditional_call,
    {
        "billing": "billing",
        "technical": "technical",
        "general": "general",
    }
)

builder.add_edge("billing", END)
builder.add_edge("technical", END)
builder.add_edge("general", END)
checkpointer = MemorySaver()
graph = builder.compile(checkpointer = checkpointer)

config = { "configurable": { "thread_id": "run-1"}}

result = graph.invoke({"query": "I was charged twice this month",
                       "category": "",
                       }, config = config)
print(result["category"])
print("\n")
print(result["response"])

Billing


pls contact the billing department
